In [1]:
import sys
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.CRITICAL, force=True, stream=sys.stdout)
# logging.getLogger('torch_numopt.numerical_optimizer').setLevel(logging.INFO)
# logging.
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
logging.getLogger("torch_numopt.numerical_optimizer").setLevel(logging.DEBUG)
# logging.getLogger('torch_numopt.numerical_optimizer').setLevel(logging.CRITICAL)
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.GradientDescent(model.parameters(), lr_method="interpolate", lr_init=1e-3)
opt = torch_numopt.NewtonLS(model.parameters(), lr_method="interpolate", lr_init=1)
# opt = torch_numopt.AdaHessianLS(model.parameters(), lr_method="lipschitz")
# opt = torch_numopt.GaussNewtonLS(model.parameters(), lr_method="quadratic")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.06378505378961563
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 1 with method interpolate and initial guess 1.
epoch:  1, loss: 0.05852882191538811
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 0.0184244 with method interpolate and initial guess 1.
epoch:  2, loss: 0.05758580192923546
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 0.00386102 with method interpolate and initial guess 1.
epoch:  3, loss: 0.055565569549798965
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 0.00851892 with method interpolate and initial guess 1.
epoch:  4, loss: 0.05225460231304169
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 0.0149388 with method interpolate and initial guess 1.
epoch:  5, loss: 0.0521608330309391
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 0.000438453 with method interpolate and initial guess 1.
epoch:  6, loss: 0.05197451636195183
INFO:torch_numopt.numerical_optimizer:Initial lr 

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.869590163230896
Test metrics:  R2 = 0.019583165645599365
